In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

# Load YOUR events DB
engine = create_engine("sqlite:///../data/events.db")
df = pd.read_sql("SELECT * FROM events", engine)
print(f"📊 {len(df)} total events ({len(df[df.event_type=='web'])} web)")


📊 22 total events (15 web)


In [2]:
def engineer_features(df):
    features = pd.DataFrame()
    ip_freq = df['source_ip'].value_counts().to_dict()
    features['ip_freq'] = df['source_ip'].map(ip_freq).fillna(1)
    features['cmd_len'] = df['command'].fillna('').str.len()
    features['sudo_flag'] = df['command'].fillna('').str.contains('sudo|root', case=False).astype(int)
    features['web_event'] = (df['event_type'] == 'web').astype(int)
    return features

X = engineer_features(df)
print("✅ Features shape:", X.shape)
print(X.describe())


✅ Features shape: (22, 4)
         ip_freq    cmd_len  sudo_flag  web_event
count  22.000000  22.000000  22.000000  22.000000
mean   11.454545  12.909091   0.272727   0.681818
std     5.413627   3.753209   0.455842   0.476731
min     1.000000   6.000000   0.000000   0.000000
25%     5.000000  13.000000   0.000000   0.000000
50%    15.000000  13.000000   0.000000   1.000000
75%    15.000000  16.000000   0.750000   1.000000
max    15.000000  18.000000   1.000000   1.000000


In [4]:
# Phase 7: Multi-class labels
y = np.zeros(len(X))
y[(X['ip_freq'] > 2) & (X['web_event'] == 1)] = 1  # Brute-force
y[(X['cmd_len'] > 15) | (X['sudo_flag'] == 1)] = 2  # Exploitation

print("🏷️ Labels:", pd.Series(y).value_counts().sort_index())

# Train Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# FIXED: Results (handles missing classes)
y_pred = model.predict(X_test)
unique_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
print("\n🎯 PHASE 7 RESULTS:")
print(classification_report(y_test, y_pred, 
                          labels=unique_classes,
                          target_names=[name for i,name in enumerate(['normal','brute-force','exploitation']) if i in unique_classes]))

# Feature Importance
print("\n📊 Top Features:")
for name, score in zip(X.columns, model.feature_importances_):
    print(f"{name}: {score:.3f}")

# SAVE MODEL (auto-loaded by main.py)
joblib.dump(model, '../honeypot_multi_model.pkl')
print("\n🚀 SAVED: honeypot_multi_model.pkl")
print("✅ Restart FastAPI → MULTI_CLASS_ML mode!")


🏷️ Labels: 0.0     1
1.0     9
2.0    12
Name: count, dtype: int64

🎯 PHASE 7 RESULTS:
              precision    recall  f1-score   support

 brute-force       1.00      1.00      1.00         3
exploitation       1.00      1.00      1.00         4

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7


📊 Top Features:
ip_freq: 0.113
cmd_len: 0.663
sudo_flag: 0.117
web_event: 0.106

🚀 SAVED: honeypot_multi_model.pkl
✅ Restart FastAPI → MULTI_CLASS_ML mode!
